In [1]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

In [2]:
def create_dataset():
    X, y = make_classification(n_samples=20, n_features=2, n_redundant=0, n_informative=2, n_clusters_per_class=1,
        random_state=42)
    return X, y

In [3]:
def initialize_weights(n):
    return np.ones(n) / n

In [4]:
def train_stump(X, y):
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X, y)
    return stump

In [5]:
def calculate_error(y_true, y_pred, weights):
    incorrect = y_true != y_pred
    error = np.sum(weights[incorrect])
    return error, incorrect

In [6]:
def calculate_alpha(error):
    error = np.clip(error, 1e-10, 1-1e-10)
    alpha = 0.5 * np.log((1-error)/error)
    return alpha

In [7]:
def update_weights(weights, incorrect, alpha):
    new_weights = weights.copy()
    new_weights[incorrect] *= np.exp(alpha)
    new_weights[~incorrect] *= np.exp(-alpha)
    return new_weights

In [8]:
def normalize_weights(weights):
    return weights / np.sum(weights)

In [9]:
def weighted_sampling(X, y, weights):
    n = len(X)
    indices = np.random.choice(np.arange(n), size=n, replace=True, p=weights)
    X_new = X[indices]
    y_new = y[indices]
    return X_new, y_new

In [10]:
def manual_adaboost(X, y, n_estimators=3):
    weights = initialize_weights(len(X))
    learners = []
    alphas = []
    for i in range(n_estimators):
        print("="*50)
        print(f"STUMP {i+1}")
        stump = train_stump(X, y)
        pred = stump.predict(X)
        error, incorrect = calculate_error(y, pred, weights)
        alpha = calculate_alpha(error)
        print("Error :", round(error,4))
        print("Alpha :", round(alpha,4))
        weights = update_weights(weights, incorrect, alpha)
        weights = normalize_weights(weights)
        print("Weights :", np.round(weights,4))
        print("Sum :", weights.sum())
        learners.append(stump)
        alphas.append(alpha)
        X, y = weighted_sampling(X, y, weights)
    return learners, alphas

In [11]:
X, y = create_dataset()
learners, alphas = manual_adaboost(X, y, n_estimators=3)

STUMP 1
Error : 0.2
Alpha : 0.6931
Weights : [0.125  0.0312 0.0312 0.125  0.0312 0.0312 0.0312 0.0312 0.0312 0.0312
 0.0312 0.0312 0.0312 0.0312 0.0312 0.125  0.0312 0.0312 0.0312 0.125 ]
Sum : 1.0
STUMP 2
Error : 0.1875
Alpha : 0.7332
Weights : [0.0769 0.0192 0.0192 0.3333 0.0192 0.0192 0.0833 0.0192 0.0192 0.0192
 0.0192 0.0192 0.0192 0.0192 0.0192 0.0769 0.0192 0.0833 0.0192 0.0769]
Sum : 1.0
STUMP 3
Error : 0.0
Alpha : 11.5129
Weights : [0.0769 0.0192 0.0192 0.3333 0.0192 0.0192 0.0833 0.0192 0.0192 0.0192
 0.0192 0.0192 0.0192 0.0192 0.0192 0.0769 0.0192 0.0833 0.0192 0.0769]
Sum : 1.0000000000000002


In [12]:
def predict(learners, alphas,X):
    final = np.zeros(len(X))
    for alpha, stump in zip(alphas, learners):
        pred = stump.predict(X)
        pred = np.where(pred==0,-1,1)
        final += alpha * pred
    final = np.sign(final)
    final = np.where(final==-1,0,1)
    return final

In [13]:
prediction = predict(learners, alphas, X)
print("Accuracy :", accuracy_score(y, prediction))

Accuracy : 0.5
